In [1]:
import sys
sys.path.append("..")
from python_editor.feature_generation_v2 import analyze_code
from python_editor.data_processing import has_executable_code
from python_editor.data_filtering import format_df, filter_df

general_code = """
''' create a user '''

import datetime
import os

class User:
    def __init__(self, name):
        self.name = name

    def get_birthday(self, birthday):
        self.birthday = datetime.datetime.strptime(birthday, "%m/%d/%Y").date()

    def days_till_birthday(self):
        today = datetime.datetime.today()

        next_birthday = datetime.datetime(today.year, self.birthday.month, self.birthday.day)
        if next_birthday < today:
            next_birthday = datetime.datetime(today.year + 1, self.birthday.month, self.birthday.day)

        return (next_birthday - today).days + 1


# get user's data   
name = input("Enter your name: ")
BirthDay = input("Enter your birthday (mm/dd/yyyy): ")

# create a user
user = User(name)
user.get_birthday(BirthDay)

# calculate days left until birthday
days_left = user.days_till_birthday()
print(f"Hello, {user.name}! You were born on {user.birthday}. You have {days_left} days left until your birthday.")
"""

In [4]:
import pandas as pd

has_executable_code(pd.Series({"text": "def hello_world():\n pass"}))

False

In [32]:
x = analyze_code(general_code)

In [33]:
x["total_names"]

16

In [34]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("tomekkorbak/python-github-code", split="test")
ds

Dataset({
    features: ['text', 'repo_name', 'path', 'language', 'license', 'size', 'score'],
    num_rows: 10000
})

In [35]:
ds.set_format("pandas")
df = ds[:]

df = format_df(df)
df = filter_df(df)

In [37]:
df

,text,repo_name,path,license,NUM_CHARS
0,from __future__ import print_function\r\n\r\ni...,Jeff-Tian/mybnb,Python27/Lib/test/test___all__.py,apache-2.0,4298
1,from __future__ import print_function\nfrom pa...,dssg/wikienergy,disaggregator/build/pandas/pandas/io/tests/tes...,mit,17790
2,"""""""\nOutlier Detection using Tukeys Filter Cla...",trademob/anna-molly,lib/plugins/tukeys_filter.py,mit,5255
3,# Copyright (c) 2018 PaddlePaddle Authors. A...,PaddlePaddle/Paddle,python/paddle/fluid/tests/unittests/xpu/test_r...,apache-2.0,6221
4,# -*- coding: utf-8 -*-\nfrom __future__ impor...,marevol/gym-starter-kit-example,pong/__init__.py,apache-2.0,240
...,...,...,...,...,...
6270,"""""""\nHere we test our output parsing and match...",PKRoma/httpie,tests/utils/matching/test_matching.py,bsd-3-clause,5827
6271,"from __future__ import absolute_import, unicod...",chrxr/wagtail,wagtail/wagtailsearch/tests/test_db_backend.py,bsd-3-clause,854
6272,from lib.action import BaseVMsAction\n\n__all_...,armab/st2contrib,packs/rackspace/actions/set_vm_metadata_item.py,apache-2.0,490
6273,#\n# Licensed to the Apache Software Foundatio...,practice-vishnoi/dev-spark-1,python/pyspark/mllib/__init__.py,apache-2.0,1208


In [39]:
df["has_executable_code"] = df.progress_apply(has_executable_code, axis=1)

100%|██████████| 6275/6275 [00:05<00:00, 1084.08it/s]


In [40]:
df["has_executable_code"].value_counts()

has_executable_code
True     5351
False     924
Name: count, dtype: int64

In [41]:
df = df[df["has_executable_code"]]

df_sample = df.sample(100)
df_sample

,text,repo_name,path,license,NUM_CHARS,has_executable_code
279,#\n# Licensed to the Apache Software Foundatio...,saltstar/spark,python/pyspark/sql/streaming.py,apache-2.0,40553,True
2440,#!/usr/bin/env python\r\nimport numpy as np\r\...,annayqho/TheCannon,code/lamost/xcalib_5labels/paper_plots/plot_su...,mit,1291,True
1487,"""""""This component provides basic support for F...",partofthething/home-assistant,homeassistant/components/foscam/camera.py,apache-2.0,8589,True
325,"""""""\nSupport for monitoring OctoPrint sensors....",persandstrom/home-assistant,homeassistant/components/sensor/octoprint.py,apache-2.0,4749,True
3543,# Copyright lowRISC contributors.\n# Licensed ...,lowRISC/opentitan,util/reggen/reg_base.py,apache-2.0,1384,True
...,...,...,...,...,...,...
5240,"#!/usr/bin/env python\n""""""This module has unit...",planetarypy/pvl,tests/test_init.py,bsd-3-clause,7666,True
348,from django.conf import settings\n\nfrom .util...,kawamon/hue,desktop/core/ext-py/django-ipware-2.1.0/ipware...,apache-2.0,2438,True
1517,"""""""Ban logic for HTTP component.""""""\nfrom coll...",PetePriority/home-assistant,homeassistant/components/http/ban.py,apache-2.0,5951,True
3505,# Copyright (c) Frederick Dean\n# See LICENSE ...,hynek/pyopenssl,tests/test_rand.py,apache-2.0,818,True


In [42]:
df_sample.to_csv("../data/test_sample.csv", index=False)